# 02 · Dati, identità e applicazioni

Notebook che prepara contenuti e identità della demo dopo `01-infrastruttura.ipynb`. Rilegge la configurazione dagli output del deployment ARM e risolve gli altri valori direttamente da Azure.

| Passo | Use case abilitato |
| --- | --- |
| 1. SQL ricambi | MCP ricambi (seed dal client solo con SQL pubblico). |
| 2. Manuali su Blob | Knowledge source dei manuali e RAG classico. |
| 3. Gruppi Entra e ACL ADLS | Demo ACL: *Tecnici* e *Direzione*, permessi per cartella. |
| 4. Login della chat | App registration, federated credential e Easy Auth senza segreti. |
| 5. Build e deploy app | Server MCP ricambi e chat React/FastAPI su App Service. |
| 6. Verifiche | Health MCP, `tools/list`, chat protetta. |

Ogni passo ha un flag `RUN_*` ed è idempotente.

In [ ]:
# Configurazione: usa lo stesso ENVIRONMENT_NAME di 01-infrastruttura.ipynb.
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: serve infra/main.bicep.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from notebook_env import load_azd_env

LOAD_AZD_ENV = False               # True = precompila con l'ambiente azd
AZD_ENVIRONMENT_NAME = ""           # vuoto = ambiente selezionato; oppure un nome esplicito
azd_values = load_azd_env(AZD_ENVIRONMENT_NAME, root=ROOT, override=True) if LOAD_AZD_ENV else {}
if LOAD_AZD_ENV:
    missing = [k for k in ("AZURE_ENV_NAME", "AZURE_SUBSCRIPTION_ID", "AZURE_TENANT_ID") if not azd_values.get(k)]
    if missing:
        raise RuntimeError(f"Valori mancanti nell'ambiente azd: {', '.join(missing)}.")

SUBSCRIPTION_ID = azd_values.get("AZURE_SUBSCRIPTION_ID", "")
ENVIRONMENT_NAME = azd_values.get("AZURE_ENV_NAME") or "wpc-demo"
DEPLOYMENT_NAME = azd_values.get("AZURE_DEPLOYMENT_NAME") or f"wpc-foundryiq-{ENVIRONMENT_NAME}"

ACL_GROUP_TECNICI = "WPC FoundryIQ ACL - Tecnici"
ACL_GROUP_DIREZIONE = "WPC FoundryIQ ACL - Direzione"
CHAT_APP_REGISTRATION = f"wpc-foundryiq-chat-{ENVIRONMENT_NAME}"

RUN_SQL_SEED = True        # usato solo se il deployment ha SQL pubblico
RUN_UPLOAD_MANUALS = True
RUN_ACL_SETUP = True       # richiede il permesso di creare gruppi Entra
RUN_CHAT_AUTH = True       # richiede il permesso di creare app registration
RUN_APP_DEPLOY = True      # richiede Node.js/npm per la build del frontend

In [ ]:
import json
import re
import shutil
import subprocess
import time
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: servono infra/, data/, agents/ e le app.")
AZ = shutil.which("az")
if AZ is None:
    raise RuntimeError("Azure CLI non trovata nel PATH: installala, esegui 'az login' e riavvia il kernel.")


def az(*args, output=True, check=True):
    """Esegue Azure CLI. Con output=True restituisce il JSON decodificato; con check=False None in caso di errore."""
    command = [AZ, *args, "--only-show-errors"] + (["--output", "json"] if output else [])
    proc = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        if check:
            raise RuntimeError(f"az {' '.join(args[:3])} fallito: {proc.stderr.strip()[-2000:]}")
        return None
    text = proc.stdout.strip()
    return (json.loads(text) if text else None) if output else text


def load_outputs(deployment_name):
    """Legge gli output del deployment ARM di subscription creato da 01-infrastruttura.ipynb."""
    deployment = az("deployment", "sub", "show", "--name", deployment_name, check=False)
    if not deployment:
        raise RuntimeError(f"Deployment '{deployment_name}' non trovato nella subscription attiva: esegui 01-infrastruttura.ipynb.")
    state = deployment["properties"].get("provisioningState")
    if state != "Succeeded":
        raise RuntimeError(f"Deployment '{deployment_name}' in stato {state}: correggi e riesegui 01-infrastruttura.ipynb.")
    return {name.upper(): item["value"] for name, item in (deployment["properties"].get("outputs") or {}).items()}


if SUBSCRIPTION_ID:
    az("account", "set", "--subscription", SUBSCRIPTION_ID, output=False)
account = az("account", "show", check=False)
if not account:
    raise RuntimeError("Azure CLI non autenticata: esegui 'az login' (eventualmente con --tenant).")
if azd_values.get("AZURE_TENANT_ID") and azd_values["AZURE_TENANT_ID"].casefold() != account["tenantId"].casefold():
    raise RuntimeError("Il tenant dell'ambiente azd non corrisponde alla subscription selezionata in Azure CLI.")
SUBSCRIPTION_ID, TENANT_ID = account["id"], account["tenantId"]
out = load_outputs(DEPLOYMENT_NAME)
if out["AZURE_TENANT_ID"] != TENANT_ID:
    raise RuntimeError(f"Il deployment usa il tenant {out['AZURE_TENANT_ID']}, Azure CLI il tenant {TENANT_ID}.")
RESOURCE_GROUP = out["AZURE_RESOURCE_GROUP"]
print(f"Subscription: {account['name']} ({SUBSCRIPTION_ID})")
print(f"Deployment:   {DEPLOYMENT_NAME} -> resource group {RESOURCE_GROUP}")

TRANSIENT = re.compile(r"\b(401|403|404|408|409|429|500|502|503|504)\b|Unauthorized|Forbidden|AuthorizationPermissionMismatch|"
                       r"AuthorizationFailed|TooManyRequests|ServiceUnavailable|timed? ?out|temporarily|Error executing tool", re.I)


def retry(label, action, attempts=8, delay=20):
    """Ripete gli errori transitori: subito dopo il deploy i ruoli RBAC impiegano minuti a propagarsi."""
    for attempt in range(1, attempts + 1):
        try:
            return action()
        except Exception as exc:  # noqa: BLE001 - riprova solo gli errori transitori riconosciuti
            if attempt == attempts or not TRANSIENT.search(str(exc)):
                raise
            print(f"  {label}: errore transitorio ({attempt}/{attempts}), riprovo tra {delay} s...")
            time.sleep(delay)

import requests
from azure.identity import AzureCliCredential

credential = AzureCliCredential(tenant_id=TENANT_ID)
me = az("ad", "signed-in-user", "show")
summary = []
print(f"Utente: {me['userPrincipalName']}")

## 1 · Database ricambi

Con il private endpoint (default) il server MCP crea lo schema e carica `mcp-ricambi/data/inventory.json` da solo, tramite la propria managed identity. Con SQL pubblico la cella apre il firewall al tuo IP, carica i dati e concede `db_datareader` all'identità del server MCP (serve `pip install mssql-python`).

In [ ]:
if out.get("SQL_PRIVATE_ENDPOINT", True):
    summary.append(("SQL", "private endpoint: dati caricati dal server MCP"))
    print("SQL privato: nessuna azione dal client.")
elif not RUN_SQL_SEED:
    summary.append(("SQL", "saltato (RUN_SQL_SEED=False)"))
else:
    import sys
    import uuid

    client_ip = requests.get("https://api.ipify.org", timeout=30).text.strip()
    az("sql", "server", "firewall-rule", "create", "--resource-group", RESOURCE_GROUP, "--server", out["SQL_SERVER_NAME"],
       "--name", "notebook-client", "--start-ip-address", client_ip, "--end-ip-address", client_ip)
    sys.path.insert(0, str(ROOT / "mcp-ricambi"))
    import sql_source  # data layer dell'app MCP: stesso schema usato in produzione

    data = json.loads((ROOT / "mcp-ricambi" / "data" / "inventory.json").read_text(encoding="utf-8"))
    conn = retry("connessione SQL", lambda: sql_source.connect(sql_source.connection_string(
        out["SQL_SERVER_FQDN"], out["SQL_DATABASE_NAME"], "ActiveDirectoryDefault")))
    try:
        counts = sql_source.seed(conn, data)
        app_name = out["MCP_APP_NAME"]
        if not re.fullmatch(r"[A-Za-z0-9-]+", app_name):
            raise ValueError(f"Nome app non valido: {app_name}")
        client_id = az("ad", "sp", "show", "--id", out["MCP_APP_PRINCIPAL_ID"])["appId"]
        sid = "0x" + uuid.UUID(client_id).bytes_le.hex().upper()
        cursor = conn.cursor()
        cursor.execute(f"IF NOT EXISTS (SELECT 1 FROM sys.database_principals WHERE name = N'{app_name}') "
                       f"CREATE USER [{app_name}] WITH SID = {sid}, TYPE = E;")
        cursor.execute(f"IF IS_ROLEMEMBER('db_datareader', N'{app_name}') = 0 ALTER ROLE db_datareader ADD MEMBER [{app_name}];")
        conn.commit()
    finally:
        conn.close()
    summary.append(("SQL", ", ".join(f"{k}={v}" for k, v in counts.items()) + f"; {app_name} in db_datareader"))

## 2 · Manuali delle macchine su Blob Storage

Carica `data/machinespecs` nel container indicizzato dalla knowledge source `machinespecs-ks` (e quindi dall'indice usato dagli agenti RAG classico). Usa il tuo login Entra: il ruolo *Storage Blob Data Contributor* assegnato dal Bicep può richiedere qualche minuto.

In [ ]:
if RUN_UPLOAD_MANUALS:
    from azure.storage.blob import BlobServiceClient

    files = sorted(p for p in (ROOT / "data" / "machinespecs").iterdir() if p.is_file())
    with BlobServiceClient(out["FOUNDRYIQ_DEMO_BLOB_ENDPOINT"], credential=credential) as blob_service:
        container = blob_service.get_container_client(out["FOUNDRYIQ_DEMO_BLOB_CONTAINER"])

        def upload_manuals():
            for path in files:
                with path.open("rb") as stream:
                    container.upload_blob(path.name, stream, overwrite=True)

        retry("upload manuali", upload_manuals)
    summary.append(("Manuali", f"{len(files)} file in {out['FOUNDRYIQ_DEMO_BLOB_CONTAINER']}"))
    print(f"Caricati {len(files)} file.")
else:
    summary.append(("Manuali", "saltato"))

## 3 · Gruppi Entra e ACL su ADLS Gen2

Crea (se mancano) i gruppi *Tecnici* e *Direzione*: sei owner di entrambi ma **membro solo di Tecnici**, così la demo ACL mostra un documento visibile e uno negato. Le ACL POSIX usano solo voci nominate di gruppo, le uniche che Foundry IQ ingerisce.

| Cartella | Gruppi con lettura |
| --- | --- |
| `pubblico/` | Tecnici, Direzione |
| `manutenzione-riservata/` | Tecnici |
| `direzione-riservata/` | Direzione |

In [ ]:
def build_acl(group_ids, include_default):
    """ACL POSIX: proprietario e maschera standard più una voce r-x per ogni gruppo autorizzato."""
    entries = ["user::rwx", "group::r-x", "other::---", "mask::r-x"] + [f"group:{g}:r-x" for g in group_ids]
    if include_default:
        entries += [f"default:{entry}" for entry in entries]
    return ",".join(entries)


ACL_FOLDERS = {
    "pubblico": ["tecnici", "direzione"],
    "manutenzione-riservata": ["tecnici"],
    "direzione-riservata": ["direzione"],
}

In [ ]:
def ensure_group(display_name, nickname, add_member):
    group = next((g for g in az("ad", "group", "list", "--display-name", display_name) if g["displayName"] == display_name), None)
    if group is None:
        group = az("ad", "group", "create", "--display-name", display_name, "--mail-nickname", nickname,
                   "--description", "Gruppo della demo ACL Foundry IQ (WPC 2026).")
        print(f"  creato: {display_name}")
    if me["id"] not in {o["id"] for o in az("ad", "group", "owner", "list", "--group", group["id"])}:
        az("ad", "group", "owner", "add", "--group", group["id"], "--owner-object-id", me["id"], output=False)
    is_member = az("ad", "group", "member", "check", "--group", group["id"], "--member-id", me["id"])["value"]
    if add_member and not is_member:
        az("ad", "group", "member", "add", "--group", group["id"], "--member-id", me["id"], output=False)
        is_member = True
    if not add_member and is_member:
        print(f"  attenzione: sei membro di '{display_name}', quindi vedrai anche i suoi documenti riservati.")
    return group["id"], is_member


if RUN_ACL_SETUP:
    from azure.storage.filedatalake import DataLakeServiceClient

    groups = {}
    groups["tecnici"], _ = ensure_group(ACL_GROUP_TECNICI, "wpc-foundryiq-acl-tecnici", add_member=True)
    groups["direzione"], _ = ensure_group(ACL_GROUP_DIREZIONE, "wpc-foundryiq-acl-direzione", add_member=False)

    def apply_acl():
        with DataLakeServiceClient(out["FOUNDRYIQ_ACL_DFS_ENDPOINT"], credential=credential) as service:
            fs = service.get_file_system_client(out["FOUNDRYIQ_ACL_CONTAINER"])
            fs.get_directory_client("/").set_access_control(acl=build_acl(groups.values(), include_default=False))
            for folder, names in ACL_FOLDERS.items():
                ids = [groups[n] for n in names]
                directory = fs.get_directory_client(folder)
                directory.create_directory()
                directory.set_access_control(acl=build_acl(ids, include_default=True))
                for path in sorted(p for p in (ROOT / "data" / "acl" / folder).iterdir() if p.is_file()):
                    file_client = directory.get_file_client(path.name)
                    file_client.upload_data(path.read_bytes(), overwrite=True)
                    file_client.set_access_control(acl=build_acl(ids, include_default=False))
                print(f"  {folder}/ -> {', '.join(names)}")

    retry("ACL ADLS", apply_acl)
    summary.append(("ACL", f"gruppi {', '.join(groups)}; {len(ACL_FOLDERS)} cartelle con ACL"))
else:
    summary.append(("ACL", "saltato"))

## 4 · Login Entra della chat

Configura Easy Auth senza segreti: app registration con `User.Read` e `user_impersonation` su Azure AI Search, claim dei gruppi di sicurezza (mostrati nel profilo), **federated credential** con la managed identity della web app e `authsettingsV2` con token store. Il token Search dell'utente è quello che la chat inoltra a Foundry IQ per filtrare i documenti.

In [ ]:
if RUN_CHAT_AUTH:
    import os
    import tempfile

    def json_file(value):
        handle, path = tempfile.mkstemp(suffix=".json")
        with os.fdopen(handle, "w", encoding="utf-8") as stream:
            json.dump(value, stream)
        return path

    redirect_uri = out["CHAT_APP_URL"].rstrip("/") + "/.auth/login/aad/callback"
    apps = az("ad", "app", "list", "--display-name", CHAT_APP_REGISTRATION)
    if apps:
        app_id = apps[0]["appId"]
        az("ad", "app", "update", "--id", app_id, "--web-redirect-uris", redirect_uri, "--enable-id-token-issuance", "true", output=False)
    else:
        app_id = az("ad", "app", "create", "--display-name", CHAT_APP_REGISTRATION, "--sign-in-audience", "AzureADMyOrg",
                    "--web-redirect-uris", redirect_uri, "--enable-id-token-issuance", "true")["appId"]
    access = [{"resourceAppId": "00000003-0000-0000-c000-000000000000",
               "resourceAccess": [{"id": "e1fe6dd8-ba31-4d61-89e7-88639da4683d", "type": "Scope"}]}]
    search_sp = az("ad", "sp", "show", "--id", "880da380-985e-4198-81b9-e05b1cc53158", check=False)
    scope_id = next((s["id"] for s in (search_sp or {}).get("oauth2PermissionScopes", []) if s["value"] == "user_impersonation"), None)
    if scope_id:
        access.append({"resourceAppId": "880da380-985e-4198-81b9-e05b1cc53158", "resourceAccess": [{"id": scope_id, "type": "Scope"}]})
    temp_files = [json_file(access)]
    try:
        az("ad", "app", "update", "--id", app_id, "--required-resource-accesses", f"@{temp_files[0]}", output=False)
        az("ad", "app", "update", "--id", app_id, "--set", "groupMembershipClaims=SecurityGroup", output=False)
        if not az("ad", "sp", "show", "--id", app_id, check=False):
            az("ad", "sp", "create", "--id", app_id)
        credentials = az("ad", "app", "federated-credential", "list", "--id", app_id)
        if not any(c["name"] == "webapp-chat-managed-identity" for c in credentials):
            temp_files.append(json_file({"name": "webapp-chat-managed-identity",
                                         "issuer": f"https://login.microsoftonline.com/{TENANT_ID}/v2.0",
                                         "subject": out["CHAT_APP_AUTH_IDENTITY_PRINCIPAL_ID"],
                                         "audiences": ["api://AzureADTokenExchange"]}))
            az("ad", "app", "federated-credential", "create", "--id", app_id, "--parameters", f"@{temp_files[-1]}")
    finally:
        for path in temp_files:
            os.remove(path)

    auth_settings = {"properties": {
        "platform": {"enabled": True, "runtimeVersion": "~1"},
        "globalValidation": {"requireAuthentication": True, "unauthenticatedClientAction": "RedirectToLoginPage",
                             "redirectToProvider": "azureactivedirectory", "excludedPaths": ["/healthz"]},
        "identityProviders": {"azureActiveDirectory": {
            "enabled": True,
            "registration": {"openIdIssuer": f"https://login.microsoftonline.com/{TENANT_ID}/v2.0", "clientId": app_id,
                             "clientSecretSettingName": "OVERRIDE_USE_MI_FIC_ASSERTION_CLIENTID"},
            "login": {"loginParameters": ["scope=openid profile email offline_access https://search.azure.com/user_impersonation"]},
            "validation": {"allowedAudiences": [app_id, f"api://{app_id}"]}}},
        "login": {"tokenStore": {"enabled": True}},
        "httpSettings": {"requireHttps": True}}}
    url = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/"
           f"Microsoft.Web/sites/{out['CHAT_APP_NAME']}/config/authsettingsV2?api-version=2024-04-01")
    token = credential.get_token("https://management.azure.com/.default").token
    requests.put(url, json=auth_settings, headers={"Authorization": f"Bearer {token}"}, timeout=120).raise_for_status()
    consent = az("ad", "app", "permission", "admin-consent", "--id", app_id, output=False, check=False)
    summary.append(("Login chat", f"appId {app_id}; admin consent {'ok' if consent is not None else 'non concesso (consenso al primo accesso)'}"))
    print("Rieseguendo 01-infrastruttura il client ID viene ritrovato per nome e passato al Bicep.")
else:
    summary.append(("Login chat", "saltato"))

## 5 · Build e deploy delle applicazioni

- **mcp-ricambi**: server MCP FastMCP con i dati di magazzino;
- **webapp-chat**: frontend React/TypeScript compilato con Vite e backend FastAPI (`app.py`).

I pacchetti zip sono creati in una cartella temporanea e pubblicati con `az webapp deploy`; App Service installa le dipendenze Python.

In [ ]:
import zipfile

EXCLUDED_DIRS = {".venv", "__pycache__", ".pytest_cache", "tests", "node_modules"}
EXCLUDED_FILES = {".env", "Dockerfile", ".dockerignore", ".webappignore", "README.md"}


def build_zip(source, include=None):
    """Crea lo zip dell'app escludendo ambienti locali, test, segreti e file di build intermedi."""
    import tempfile

    target = Path(tempfile.mkdtemp()) / f"{source.name}.zip"
    with zipfile.ZipFile(target, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(source.rglob("*")):
            rel = path.relative_to(source)
            if path.is_dir() or EXCLUDED_DIRS.intersection(rel.parts) or path.name in EXCLUDED_FILES or path.suffix == ".pyc":
                continue
            if include and not any(rel.as_posix() == i or rel.as_posix().startswith(i + "/") for i in include):
                continue
            archive.write(path, rel.as_posix())
    return target


if RUN_APP_DEPLOY:
    npm = shutil.which("npm")
    if npm is None:
        raise RuntimeError("npm non trovato: installa Node.js 20+ per compilare il frontend.")
    frontend = ROOT / "webapp-chat" / "frontend"
    for step in (["ci"], ["run", "build"]):
        subprocess.run([npm, *step], cwd=frontend, check=True)
    packages = {
        out["MCP_APP_NAME"]: build_zip(ROOT / "mcp-ricambi"),
        out["CHAT_APP_NAME"]: build_zip(ROOT / "webapp-chat", include=["app.py", "mcp_references.py", "requirements.txt", "frontend/dist"]),
    }
    for app_name, package in packages.items():
        print(f"Deploy {package.name} -> {app_name} ({package.stat().st_size // 1024} KB)...")
        az("webapp", "deploy", "--resource-group", RESOURCE_GROUP, "--name", app_name, "--src-path", str(package),
           "--type", "zip", output=False)
        shutil.rmtree(package.parent, ignore_errors=True)
    summary.append(("App", ", ".join(packages)))
else:
    summary.append(("App", "deploy saltato"))

## 6 · Verifiche

La chiave MCP è letta dalle impostazioni dell'App Service solo per `tools/list` e poi rimossa dalla memoria; non viene stampata.

In [ ]:
def mcp_json(response):
    """Le risposte MCP streamable HTTP possono essere JSON o Server-Sent Events."""
    response.raise_for_status()
    if "text/event-stream" in response.headers.get("content-type", ""):
        data = [line[5:].strip() for line in response.text.splitlines() if line.startswith("data:")]
        return json.loads(data[-1])
    return response.json()


def health():
    response = requests.get(out["MCP_HEALTH_URL"], timeout=60)
    response.raise_for_status()
    return response.json()


status = retry("health MCP", health, attempts=20, delay=30)
inventory = status.get("inventory") or {}
summary.append(("Health MCP", f"{status.get('status')}; sorgente {inventory.get('source')}; ricambi {inventory.get('parts')}"))

settings = az("webapp", "config", "appsettings", "list", "--resource-group", RESOURCE_GROUP, "--name", out["MCP_APP_NAME"])
mcp_key = next(s["value"] for s in settings if s["name"] == "MCP_API_KEY")
del settings
try:
    tools = mcp_json(requests.post(out["MCP_SERVER_URL"], timeout=60,
                                   headers={"x-api-key": mcp_key, "Accept": "application/json, text/event-stream"},
                                   json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}))
finally:
    mcp_key = None
summary.append(("MCP tools/list", ", ".join(t["name"] for t in tools.get("result", {}).get("tools", []))))

reply = requests.get(out["CHAT_APP_URL"] + "/api/config", timeout=60, allow_redirects=False)
summary.append(("Chat protetta", f"HTTP {reply.status_code} senza login ({'OK' if reply.status_code in (302, 401) else 'verifica'})"))

import pandas as pd
from IPython.display import display

display(pd.DataFrame(summary, columns=["passo", "esito"]))
print("Prossimo passo: notebooks/deploy/03-foundry-iq-agenti.ipynb")